# 0. Entorno y GPU

Este libro se corre en el ambiente de Miniconda **`deeplearning`** (Python 3.10, PyTorch con CUDA).
En VS Code: *Select Kernel* → *Jupyter Kernel* → `deeplearning`. En Jupyter: menú *Kernel* →
*Change kernel* → `deeplearning`.

**Uso de la GPU.** En Windows no existe RAPIDS cuML (el scikit-learn en GPU solo corre en Linux o
WSL). Los modelos del curso se ejecutan en la GPU con PyTorch y XGBoost (clases de
`mirage/ml/gpu.py`, con la interfaz de scikit-learn). Lo que queda en CPU: el preprocesamiento
(imputar, escalar, codificar, que tarda milisegundos) y los modelos de series de tiempo ARIMA, VAR y
Prophet, que no tienen versión en GPU y trabajan con 127 datos mensuales.

In [1]:
import sys, warnings, json, time
from pathlib import Path
ML = Path.cwd().parent                      # carpeta ml/
sys.path.insert(0, str(ML))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
PROCESSED, REPORTS = ML / "data" / "processed", ML / "reports"
PRED = REPORTS / "predicciones"; PRED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", palette="deep")

In [2]:
import subprocess, platform, sklearn, scipy, statsmodels
print("Python", platform.python_version(), "| ejecutable:", sys.executable)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,utilization.gpu",
                      "--format=csv"], capture_output=True, text=True).stdout)

Python 3.10.20 | ejecutable: C:\miniconda3\envs\deeplearning\python.exe
name, driver_version, memory.total [MiB], utilization.gpu [%]
NVIDIA GeForce RTX 4050 Laptop GPU, 580.88, 6141 MiB, 0 %



In [3]:
import torch, xgboost
from mirage.ml import gpu as G
print("PyTorch", torch.__version__, "| CUDA disponible:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
print("XGBoost", xgboost.__version__, "| scikit-learn", sklearn.__version__, "| statsmodels", statsmodels.__version__)
print("Dispositivo que usarán los modelos:", G.device_disponible())

PyTorch 2.5.1+cu121 | CUDA disponible: True | NVIDIA GeForce RTX 4050 Laptop GPU
XGBoost 3.2.0 | scikit-learn 1.5.2 | statsmodels 0.14.6
Dispositivo que usarán los modelos: cuda


In [4]:
# Prueba rápida: una multiplicación grande en la GPU y un XGBoost en CUDA
x = torch.randn(4000, 4000, device=G.device_disponible()); t0 = time.time(); (x @ x).sum().item()
print(f"Multiplicación 4000×4000 en {G.device_disponible()}: {time.time()-t0:.3f} s")
Xp = np.random.rand(20000, 30).astype("float32"); yp = Xp @ np.random.rand(30)
m = G.xgb_boost(n_estimators=100).fit(Xp, yp)
print("XGBoost configurado en:", json.loads(m.get_booster().save_config())["learner"]["generic_param"]["device"])

Multiplicación 4000×4000 en cuda: 0.069 s


XGBoost configurado en: cuda:0


In [5]:
# Prophet necesita encontrar tbb.dll junto a su binario (problema conocido en Windows)
import prophet, shutil
base = Path(prophet.__file__).parent / "stan_model"
if not (base / "tbb.dll").exists():
    src = next(base.glob("cmdstan-*/stan/lib/stan_math/lib/tbb/tbb.dll"))
    shutil.copy(src, base / "tbb.dll"); print("tbb.dll copiada")
print("Prophet", prophet.__version__, "listo")

Importing plotly failed. Interactive plots will not work.


Prophet 1.5.0 listo
